# 01b — All 15 quarters in one shape

Reads the 30 Ministry of Commerce (MC) files (2023 Q1 – 2026 Q3) **once**, checks them, and saves the table
`mc_all_quarters.parquet`. The next notebooks load that file in about a minute instead of reading the Excel files again.

**Rules for this notebook**
- The raw files are never changed. Anything cleaned is saved as a new file.
- Nothing is dropped. Problem rows get a flag column instead.
- Data never goes into Git. Clear all outputs before committing.

**The preprocessing notebooks, in order** (each one loads what the one before saved):
1. `01a_id_check`: did the ID change, or only its column name? (stands alone)
2. `01b_all_quarters`: reads the 30 Excel files (2023 Q1 – 2026 Q3) once and saves `mc_all_quarters.parquet`
3. `01c_id_matching`: one ID per business, saves `mc_matched.parquet`
4. `01d_survival_label`: the one-year label, saves `labels.parquet`
5. `01e_ssic_classification`: SSIC from the name, saves `ssic_by_business.csv`, `business_names.csv`, `business_city.csv`
6. `01f_old_data_branches`: old data (2023 Q1 – 2025 Q2), every branch joined to its main (stands alone)

### Cell 1 — Setup
Finds the data. On Colab it downloads the raw files from the team's Google Drive folder (temporary copies,
gone when the session ends). On a laptop it reads the local copy in `DATA_DIR`.

The results that the next notebook loads go in `SAVED_DIR`: the folder **saved** inside the team's shared Drive
folder **Darb DataSets**. Everyone loads the same results, and they are still there in a new Colab session.
On Colab: add a shortcut to Darb DataSets in your My Drive first (Drive → right-click the folder → Organize →
Add shortcut → My Drive). Colab asks once to connect Drive.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    !pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
    from google.colab import drive
    drive.mount('/content/drive')
    TEAM_DIR = Path('/content/drive/MyDrive/Darb DataSets')     # the team's shared folder (a shortcut in My Drive)
    if not TEAM_DIR.exists():
        raise FileNotFoundError('Darb DataSets is not in your My Drive: add a shortcut to it (see the text above), then run this cell again.')
    SAVED_DIR = TEAM_DIR / 'saved'                              # shared with the team, kept between Colab sessions
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', r'C:\Users\Nasser Alyahya\darb-data'))
    SAVED_DIR = DATA_DIR / 'saved'

MC_DIR = DATA_DIR / 'MC'
SAVED_DIR.mkdir(parents=True, exist_ok=True)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR, '| saved results:', SAVED_DIR)

### Cell 1b — Read an MC file (Excel only the first time)
Reading the big Excel files is the slow part. The first time a file is read, an **exact copy** of it (every row
and every column, all as text) is saved in `SAVED_DIR/mc_excel_copies`. After that the copy is read instead,
in seconds, and on Colab the Excel file is not downloaded again. Delete a copy to read its Excel file again.

In [ ]:
import pandas as pd

EXCEL_COPIES = SAVED_DIR / 'mc_excel_copies'
EXCEL_COPIES.mkdir(parents=True, exist_ok=True)

def has_copy(name):
    return (EXCEL_COPIES / f'{name}.parquet').exists()

def read_mc(name, usecols=None):
    # One MC file, all as text: from its saved copy if there is one, otherwise from Excel (and the copy is saved)
    copy = EXCEL_COPIES / f'{name}.parquet'
    if copy.exists():
        return pd.read_parquet(copy, columns=usecols)
    table = pd.read_excel(MC_DIR / name, engine='calamine', dtype=str)
    if all(isinstance(c, str) for c in table.columns):     # parquet needs text column titles
        temp = copy.with_suffix('.part')                     # a copy cut off halfway is never used
        table.to_parquet(temp, index=False)
        temp.rename(copy)
    return table[usecols] if usecols else table

### Cell 8 — Read every quarter (2023 Q1 – 2026 Q3)
Reads all 30 files (15 quarters, 2 parts each) with **all** their columns. Column names change over the years,
so each one gets a single common name (for example "تاريخ الإصدار (ميلادي)" and "تاريخ انشاء السجل" both become
`start_date`). Two columns say where a row came from: `quarter` and `source_file`.

2025 Q3 is read too. Its PART1 has no ID, so those rows get the flag `flag_no_id`.
2023 Q3 is read too, but it is incomplete (its PART1 has only about 10,000 rows).
2023 Q4, 2024 Q1 and 2024 Q2 are not in the Drive yet: on Colab, upload them to `MC` first.

**Three fixes while reading:**
1. A row whose values are the column titles (the header copied as a data row, found in 2026 Q2 PART1) is
   **removed**: it is not a business. The number removed is printed.
2. In 2025 Q4 the column "الكيان القانوني" holds the **detailed legal form** (32 values), not the entity type.
   In that quarter it becomes `legal_form`, and `entity_type` is filled from it: "مؤسسة…" → مؤسسة,
   "شركة…" → شركة, "فرع …" kept as it is. The rule is checked on 2026 Q1, where both columns exist.
3. In **2023 Q3 PART1** the region values sit under the e-commerce title, and the e-commerce values under
   "لديه تجارة إلكترونية؟", so that file's titles are corrected (same fix as `01f_old_data_branches`).

Nothing else is deleted.
The first run reads the Excel files (about 10–15 minutes on Colab). After that it reads the saved copies (Cell 1b) and takes a few minutes.

In [ ]:
import re
import numpy as np
import pandas as pd

# Every quarter, 2023 Q1 – 2026 Q3: (quarter, file name, Google Drive ID; None = not in the Drive yet)
ALL_FILES = [
    ('2023Q1', 'CR_2023-01q-active-crs_PART1.xlsx',   '1R-opbDgZDivW63kxt-WVe2G7x_zaDRym'),
    ('2023Q1', 'CR_2023-01q-active-crs_PART2.xlsx',   '1VgvjY6KwmqajrbGqOU9b95k9c5pFa18P'),
    ('2023Q2', 'CR_2023-02q-active-crs_PART1.xlsx',   '10rme8h_C8pPJUB-LHNfy2HlQUWHxLm0_'),
    ('2023Q2', 'CR_2023-02q-active-crs_PART2.xlsx',   '1_r4EBy4KJIwNgIJJnNPkMgQ3r_Vk4t9R'),
    ('2023Q3', 'CR_2023-03q-active-crs_PART1.xlsx',   '1lZ9kznVbx5eOgLSBDyGdOozQ-XoXmhbQ'),
    ('2023Q3', 'CR_2023-03q-active-crs_PART2.xlsx',   '1d03GvYu_035nXsmStTXoUVLm-4ybJukh'),
    ('2023Q4', 'CR_2023-04q-active-crs_PART1.xlsx',   None),
    ('2023Q4', 'CR_2023-04q-active-crs_PART2.xlsx',   None),
    ('2024Q1', 'CR_2024-01q-active-crs_PART1.xlsx',   None),
    ('2024Q1', 'CR_2024-01q-active-crs_PART2.xlsx',   None),
    ('2024Q2', 'CR_2024-02q-active-crs_PART1.xlsx',   None),
    ('2024Q2', 'CR_2024-02q-active-crs_PART2.xlsx',   None),
    ('2024Q3', 'CR_2024-03q-active-crs_PART1-2.xlsx', '1YAjKdTEzTf66Z_wK0nNVAboD_NoskbKV'),
    ('2024Q3', 'CR_2024-03q-active-crs_PART2-2.xlsx', '1bCd3HFyhCk9VbZgT79wxsIxgBBflKps0'),
    ('2024Q4', 'CR_2024-04q-active-crs_PART1.xlsx',   '1GPJUCZ5HiTa-_ZJBVLjrp58Ha53nOBGN'),
    ('2024Q4', 'CR_2024-04q-active-crs_PART2.xlsx',   '10y-NeQI6P78Sb_b3mqjTjcdpZq9zNvNK'),
    ('2025Q1', 'CR_2025-01q-active-crs_PART1.xlsx',   '1QTeifZpbmotbWmbZoliWoVcDxSK8y8uf'),
    ('2025Q1', 'CR_2025-01q-active-crs_PART2.xlsx',   '12LqZsJ8WHI4qUza_mXP4iUQPuwG9x_2d'),
    ('2025Q2', '2025-02q-active-crs_PART1-2.xlsx',    '1pAlWmlVTGwCY2XnwhuujvFz9w8ys3nfp'),
    ('2025Q2', '2025-02q-active-crs_PART2.xlsx',      '14LNdY7UCE-itRV4ubyfRzapGtwgoDda_'),
    ('2025Q3', '2025-03q-active-crs_PART1.xlsx',      '11TR0gdOc1u486yR8y9jYkA5jXqwLwxVo'),
    ('2025Q3', '2025-03q-active-crs_PART2-2.xlsx',    '1K4jyfIVbWfALc8bdB0-agPAwBev6SqFD'),
    ('2025Q4', '2025-04q-active-crs_PART1.xlsx',      '1tzttjeDaz8WQrFSGE7WYI5AMQiNSsRrM'),
    ('2025Q4', '2025-04q-active-crs_PART2.xlsx',      '1rqw-5VqvreNlA7hJN4sY3wOzgmOI485a'),
    ('2026Q1', '2026 01q active_PART1.xlsx',          '1uXVoAo8IoDVhZEBJcZEdq1T3Rhfkw9kZ'),
    ('2026Q1', '2026 01q active_PART2.xlsx',          '1YukIJGFb8GIBxV5rCxXmXN023kU6bjr2'),
    ('2026Q2', '2026 02q active crs_PART1.xlsx',      '1KBjgL6FXdI-aeWumy15rnhlE20_W0ZNJ'),
    ('2026Q2', '2026 02q active crs_PART2.xlsx',      '1BrmeQHLyxET9xPulJDxBQ_68qZ8zu6kf'),
    ('2026Q3', 'CR A 2026-09-30_PART1.xlsx',          '1TGnGo3x2rkvDvbwREznik4ko0WGzXhHy'),
    ('2026Q3', 'CR A 2026-09-30_PART2-2.xlsx',        '1siJD5yNZTXRINRazJP2LfpFm8xlKcVZw'),
]

# One common name for every column, whatever its name in a given year
COMMON_NAMES = {
    # ID of the record
    'الرقم الوطني': 'id',                          # up to 2025 Q3
    'الرقم الموحد': 'id',                          # from 2025 Q4
    # for a branch: the ID of its main record (only up to 2025 Q3)
    'الرقم الرئيسي للسجل التجاري': 'main_id',
    # the commercial registration number (only from 2025 Q4)
    'رقم السجل': 'cr_number',
    # main record or branch
    'نوع السجل التجاري': 'record_type',
    'نوع السجل': 'record_type',
    'اسم السجل التجاري': 'name',
    'اسم السجل': 'name',
    # establishment, company ...
    'نوع الكيان': 'entity_type',
    'الكيان القانوني': 'entity_type',
    # the detailed legal form (despite its Arabic name, this is not the business activity)
    'نوع النشاط التجاري (بالعربية)': 'legal_form',
    'الكيان القانوني.1': 'legal_form',
    'رأس المال': 'capital',
    'اسم المدينة (بالعربية)': 'city',
    'المدينة': 'city',
    'اسم المنطقة (بالعربية)': 'region',
    'المنطقة': 'region',
    'مفصح بالتجارة الإلكترونية؟': 'ecommerce',
    'مفصح بتجارة إلكترونية؟': 'ecommerce',
    'مفصح تجارة إلكترونية؟': 'ecommerce',
    'لديه تجارة إلكترونية؟': 'ecommerce',            # 2023
    'تاريخ الإصدار (ميلادي)': 'start_date',
    'تاريخ انشاء السجل': 'start_date',
}
ORDER = ['quarter', 'source_file', 'id', 'main_id', 'cr_number', 'record_type', 'name', 'entity_type',
         'legal_form', 'capital', 'region', 'city', 'ecommerce', 'start_date']
# Columns with few different values are stored as categories, and text in a compact format, to save memory
FEW_VALUES = ['quarter', 'source_file', 'record_type', 'entity_type', 'legal_form', 'region', 'city', 'ecommerce']

# Exceptions to COMMON_NAMES for one quarter
QUARTER_NAMES = {
    # in 2025 Q4 "الكيان القانوني" holds the detailed legal form (32 values), not the entity type
    '2025Q4': {'الكيان القانوني': 'legal_form'},
}
# Exceptions for one file: 2023 Q3 PART1 has no region title. Its region values are under
# "مفصح بالتجارة الإلكترونية؟" and its e-commerce values under "لديه تجارة إلكترونية؟"
FILE_FIXES = {'CR_2023-03q-active-crs_PART1.xlsx': {'مفصح بالتجارة الإلكترونية؟': 'region',
                                                    'لديه تجارة إلكترونية؟': 'ecommerce'}}

def entity_type_from(legal_form):
    """Entity type from the detailed legal form: 'مؤسسة…' -> مؤسسة, 'شركة…' -> شركة, 'فرع …' kept as it is.
    Anything else stays empty (shown by the check at the end of this cell)."""
    form = legal_form.astype('string').fillna('')
    kind = np.select([form.str.startswith('فرع'), form.str.startswith('مؤسس'), form.str.startswith('شركة')],
                     [form, 'مؤسسة', 'شركة'], default='')
    return pd.Series(kind, index=legal_form.index, dtype='string').replace('', pd.NA)

if IN_COLAB:
    import gdown
    MC_DIR.mkdir(parents=True, exist_ok=True)
    for quarter, name, drive_id in ALL_FILES:
        if not (MC_DIR / name).exists() and not has_copy(name):
            if drive_id is None:
                print('not in the Drive yet, upload it to MC first:', name)
                continue
            print('downloading', name)
            gdown.download(id=drive_id, output=str(MC_DIR / name), quiet=True)

unknown_columns = set()
header_copies = {}
parts = []
for quarter, name, _ in ALL_FILES:
    part = read_mc(name)
    for column in part.columns:
        part[column] = part[column].str.strip()
    # Fix 1: a row whose values are the column titles is the header copied as data, not a business
    titles = pd.Series([re.sub(r'\.\d+$', '', str(c)).strip() for c in part.columns], index=part.columns)
    is_title = part.eq(titles).sum(axis=1) >= len(titles) // 2
    if is_title.any():
        header_copies[name] = int(is_title.sum())
        part = part[~is_title]
    names = {**COMMON_NAMES, **QUARTER_NAMES.get(quarter, {}), **FILE_FIXES.get(name, {})}
    unknown_columns |= set(part.columns) - set(names)
    part = part.rename(columns=names)
    if part.columns.duplicated().any():
        print(f'  {name}: two columns got the same name {sorted(set(part.columns[part.columns.duplicated()]))}, the first is kept')
        part = part.loc[:, ~part.columns.duplicated()]
    # Fix 2: a quarter with a legal form but no entity type (2025 Q4) gets the entity type from the legal form
    if 'entity_type' not in part.columns and 'legal_form' in part.columns:
        part['entity_type'] = entity_type_from(part['legal_form'])
    part = part.assign(quarter=quarter, source_file=name).reindex(columns=ORDER)
    for column in ORDER:
        part[column] = part[column].astype('category' if column in FEW_VALUES else 'string[pyarrow]')
    parts.append(part)
    print(f'{quarter}  {name:<40} {len(part):>10,} rows')

mc = pd.concat(parts, ignore_index=True)
for column in FEW_VALUES:
    mc[column] = mc[column].astype('category')
del parts

# Rows with no ID (2025 Q3 PART1) get a flag. Nothing is deleted.
mc['flag_no_id'] = mc['id'].isna()

print()
print(f'All quarters together: {len(mc):,} rows | memory used: {mc.memory_usage(deep=True).sum() / 1e9:.1f} GB')
print('Columns the notebook did not recognise:', sorted(unknown_columns) or 'none')
print('Header rows read as data, removed:', header_copies or 'none')

# Check of fix 2 on 2026 Q1, where both columns exist: does the rule give the real entity type?
check = mc.loc[mc['quarter'] == '2026Q1', ['legal_form', 'entity_type']].astype('string')
check['rule'] = entity_type_from(check['legal_form'])
same = (check['rule'] == check['entity_type']).fillna(False)
print()
print(f'Entity type rule checked on 2026 Q1: same as the real entity type for {same.mean():.2%} of rows')
if (~same).any():
    print('Rows where it differs (legal form, real entity type, rule):')
    print(check[~same].value_counts(dropna=False).head(10).to_string())
in_q4 = mc['quarter'] == '2025Q4'
print('2025 Q4 entity type (filled by the rule):')
print(mc.loc[in_q4, 'entity_type'].astype('string').value_counts(dropna=False).to_string())
del check, same


### Cell 9 — Health check per quarter
Only counts. For every quarter: rows, main records and branches, rows with no ID, IDs that repeat
(and how many of those appear in both PART1 and PART2, like the overlap found in 2025 Q2), and which
columns the quarter does not have at all.

In [ ]:
def quarter_health(table):
    rows = []
    for quarter, q in table.groupby('quarter', observed=True):
        with_id = q[q['id'].notna()]
        per_id = with_id.groupby('id', observed=True)['source_file'].agg(['size', 'nunique'])
        rows.append({
            'quarter': quarter,
            'rows': len(q),
            'main': (q['record_type'] == 'رئيسي').sum(),
            'branch': (q['record_type'] == 'فرعي').sum(),
            'type unknown': q['record_type'].isna().sum(),
            'no ID': q['id'].isna().sum(),
            'IDs repeated': int((per_id['size'] > 1).sum()),
            '…in both parts': int(((per_id['size'] > 1) & (per_id['nunique'] > 1)).sum()),
            'columns missing': ', '.join(c for c in ['main_id', 'cr_number', 'legal_form', 'ecommerce', 'region']
                                         if q[c].isna().all()) or '—',
        })
    return pd.DataFrame(rows).set_index('quarter')

health = quarter_health(mc)
print(health.drop(columns='columns missing').to_string())
print()
print(health[['columns missing']].to_string())


### Cell 10 — Can we trust each quarter?
Two checks that only count.

**A. Main / branch labels.** Up to 2025 Q3 a branch has a `main_id` that points to its main record. Counts how
often `record_type` agrees with that, to see whether a quarter (such as 2025 Q1, almost all "main") labels
branches as main.

**B. Completeness.** A business that is active in a later quarter and already existed before the earlier quarter
ended should also appear in the earlier quarter. The share that does = how complete the earlier quarter is.
A low share means businesses are missing from that quarter, and would wrongly look closed in the labels.

In [ ]:
QUARTERS = ['2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2',
            '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']
QUARTER_END = {'2023Q1': '2023-03-31', '2023Q2': '2023-06-30', '2023Q3': '2023-09-30', '2023Q4': '2023-12-31',
               '2024Q1': '2024-03-31', '2024Q2': '2024-06-30',
               '2024Q3': '2024-09-30', '2024Q4': '2024-12-31', '2025Q1': '2025-03-31', '2025Q2': '2025-06-30',
               '2025Q3': '2025-09-30', '2025Q4': '2025-12-31', '2026Q1': '2026-03-31', '2026Q2': '2026-06-30',
               '2026Q3': '2026-09-30'}
start_day = mc['start_date'].str[:10]          # date only; dates are written as YYYY-MM-DD, so text order = date order

# ---------- A. Is "main / branch" labelled correctly? (quarters that have main_id) ----------
print('A. Main / branch labels compared with main_id (a real branch points to another record)')
rows = []
for quarter in [q for q in QUARTERS if q <= '2025Q3']:   # the quarters that have main_id
    q = mc[mc['quarter'] == quarter]
    points_elsewhere = (q['main_id'].notna() & (q['main_id'] != q['id'])).fillna(False)
    is_main, is_branch = q['record_type'] == 'رئيسي', q['record_type'] == 'فرعي'
    rows.append({'quarter': quarter,
                 'main': int(is_main.sum()),
                 'main that points to another record': int((is_main & points_elsewhere).sum()),
                 'branch': int(is_branch.sum()),
                 'branch with no main record': int((is_branch & ~points_elsewhere).sum())})
print(pd.DataFrame(rows).set_index('quarter').to_string())

# ---------- B. How complete is each quarter? ----------
# Main records of a later quarter that already existed before the earlier quarter ended
# should (almost) all be in the earlier quarter too. The share found = how complete the earlier quarter is.
print()
print('B. Completeness of each quarter (checked against the quarter after it)')
pairs = list(zip(QUARTERS, QUARTERS[1:])) + [('2025Q2', '2025Q4')]   # 2025Q2 -> 2025Q4 skips the broken 2025Q3
rows = []
for earlier, later in pairs:
    later_main = mc[(mc['quarter'] == later) & (mc['record_type'] == 'رئيسي')]
    existed = later_main[(start_day[later_main.index] <= QUARTER_END[earlier]).fillna(False)]   # empty date = not counted
    earlier_ids = mc.loc[(mc['quarter'] == earlier) & mc['id'].notna(), 'id']
    found = existed['id'].isin(earlier_ids)
    rows.append({'earlier quarter': earlier, 'checked against': later,
                 'main records that already existed': len(existed),
                 'found in earlier quarter': int(found.sum()),
                 'completeness': f'{found.mean():.1%}'})
print(pd.DataFrame(rows).set_index('earlier quarter').to_string())


### Save — the table for the next notebooks
Saves `mc` to `mc_all_quarters.parquet` in `SAVED_DIR`. `01c_id_matching` loads it.
Run this again only when a raw file or Cell 8 changes.

In [ ]:
OUT = SAVED_DIR / 'mc_all_quarters.parquet'
mc.to_parquet(OUT, index=False)
print(f'Saved {len(mc):,} rows to {OUT} ({OUT.stat().st_size / 1e6:,.0f} MB)')